## STORE SALES- TIME SERIES FORECASTING

In [ ]:
#Bu proje, Kaggle'ın "Store Sales" veri setini kullanarak perakende mağazalarının günlük müşteri trafik hacmini tahmin etmek amacıyla geliştirilmiştir.
#Mağazaların gelecekteki günlük işlem yoğunluğunu tahmin ederek personel planlaması, lojistik ve operasyonel süreçlerin optimize edilmesini sağlamak.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# --- KESİN ÇÖZÜM: Dosya yollarını kontrol ederek yükleme ---
print("📦 Veri dosyaları kontrol ediliyor...")

# Klasördeki mevcut dosyaları listeyelim ki ismi tam görelim
mevcut_dosyalar = os.listdir('.')
zip_ismi = [f for f in mevcut_dosyalar if 'transactions' in f and f.endswith('.zip')]

if len(zip_ismi) > 0:
    transactions_path = zip_ismi[0]
    print(f"✅ Bulunan İşlem Verisi: {transactions_path}")
else:
    # Eğer zip yoksa direkt csv aranır
    transactions_path = 'transactions.csv'
    print("⚠️ Zip dosyası bulunamadı, düz CSV aranıyor...")

# Verileri güvenli yollardan yükleme
transactions = pd.read_csv(transactions_path)
stores = pd.read_csv('stores.csv')
print("✅ Veriler hatasız yüklendi!")

# --- TEMİZLİK ---
print("🔄 Eski model dosyaları temizleniyor...")
for file in ['store_transactions_model.pkl', 'le_city.pkl', 'le_type.pkl']:
    if os.path.exists(file):
        os.remove(file)

# 1. BİRLEŞTİRME (MERGE)
df_merged = pd.merge(transactions, stores, on='store_nbr', how='left')
df_merged['date'] = pd.to_datetime(df_merged['date'])

# 2. ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING)
df_merged['year'] = df_merged['date'].dt.year
df_merged['month'] = df_merged['date'].dt.month
df_merged['day'] = df_merged['date'].dt.day
df_merged['dayofweek'] = df_merged['date'].dt.dayofweek
df_merged['is_weekend'] = df_merged['date'].dt.dayofweek.isin([5, 6]).astype(int)

# 3. KATEGORİK DÖNÜŞTÜRÜCÜLER
le_city = LabelEncoder()
df_merged['city'] = le_city.fit_transform(df_merged['city'])

le_type = LabelEncoder()
df_merged['type'] = le_type.fit_transform(df_merged['type'])

# 4. KESİN SÜTUN SIRALAMASI
features = ['store_nbr', 'city', 'type', 'cluster', 'year', 'month', 'day', 'dayofweek', 'is_weekend']
X = df_merged[features]
y = df_merged['transactions']

# 5. KRONOLOJİK VERİ BÖLME
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 6. MODELİ EĞİTME
print("🚀 Model eğitiliyor, lütfen bekleyin...")
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 7. DÖNÜŞTÜRÜCÜLERİ VE MODELİ KAYDETME
joblib.dump(model, 'store_transactions_model.pkl')
joblib.dump(le_city, 'le_city.pkl')
joblib.dump(le_type, 'le_type.pkl')

print("🎯 BAŞARILI! Model ve dönüştürücüler hatasız şekilde kaydedildi.")

📦 Veri dosyaları kontrol ediliyor...
⚠️ Zip dosyası bulunamadı, düz CSV aranıyor...
✅ Veriler hatasız yüklendi!
🔄 Eski model dosyaları temizleniyor...
🚀 Model eğitiliyor, lütfen bekleyin...
🎯 BAŞARILI! Model ve dönüştürücüler hatasız şekilde kaydedildi.


In [7]:
#Ham tarih verisinden periyodik döngüleri yakalamak için Yıl, Ay, Gün ve Haftanın Günü (0-6) özellikleri türetilmiştir.
#Hafta sonu yoğunluğunu modele öğretmek için is_weekend (0/1) bayrağı eklenmiştir.
#Mağaza tipi (type) ve şehir (city) gibi kategorik veriler LabelEncoder ile sayısallaştırılmıştır.

In [ ]:
#Mağaza tipi (type) ve şehir (city) gibi kategorik veriler LabelEncoder ile sayısallaştırılmıştır.